In [ ]:
有保存版本

In [1]:
import os
import time
import base64
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

save_dir = r"C:\Users\wrz\Desktop\FYPProject\数据源"
os.makedirs(save_dir, exist_ok=True)

chrome_options = Options()
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-print-preview")

service = Service(executable_path=r"C:\Program Files\Google\Chromedriver\chromedriver.exe")
driver = webdriver.Chrome(service=service, options=chrome_options)
wait = WebDriverWait(driver, 20)

def sanitize_filename(filename):
    filename = re.sub(r'\\', '&', filename)
    return re.sub(r'[/:*?"<>|]', '_', filename)

def save_pdf(driver, file_path):
    try:
        pdf_content = driver.execute_cdp_cmd("Page.printToPDF", {
            "format": "A4",
            "landscape": False,
            "printBackground": True
        })
        pdf_data = pdf_content.get('data', '')
        if not pdf_data:
            print("PDF 数据为空。")
            return
        
        with open(file_path, "wb") as f:
            f.write(base64.b64decode(pdf_data))
        print(f"PDF 已保存: {file_path}")
    except Exception as e:
        print(f"保存失败: {e}")

driver.get("https://sousuo.www.gov.cn/zcwjk/policyRetrieval")
time.sleep(3)
print(f"当前页面: {driver.current_url}")

当前页面: https://sousuo.www.gov.cn/zcwjk/policyRetrieval


In [2]:

try:
    start_date_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="开始日期"]')))
    end_date_input = driver.find_element(By.XPATH, '//input[@placeholder="结束日期"]')

    driver.execute_script("arguments[0].removeAttribute('readonly');", start_date_input)
    driver.execute_script("arguments[0].removeAttribute('readonly');", end_date_input)

    start_date_input.clear()
    start_date_input.send_keys("2025-01-01")

    end_date_input.clear()
    end_date_input.send_keys("2025-3-1")

    driver.execute_script("document.body.click();")

    search_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//button/span[contains(text(), "搜索")]')))
    search_button.click()

    print("成功触发搜索")
    time.sleep(5)
    print(f"当前页面: {driver.current_url}")
except Exception as e:
    print("设置日期筛选条件时出现错误:", e)
    driver.quit()
    exit()

成功触发搜索
当前页面: https://sousuo.www.gov.cn/zcwjk/policyRetrieval


In [ ]:

try:
    page_input = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "pnum"))
    )
    driver.execute_script("arguments[0].scrollIntoView();", page_input)  
    page_input.clear() 
    page_input.send_keys("54")  


    confirm_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.CLASS_NAME, "newsPageBtn"))
    )
    confirm_button.click()


    WebDriverWait(driver, 10).until(EC.url_changes(driver.current_url))
    print(f"成功跳转到第 54 页: {driver.current_url}")

except Exception as e:
    print("跳页失败:", e)

In [3]:
def scrape_page():
    while True:
        try:
            print("爬取当前页面的所有文件...")

            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located(
                    (By.XPATH, "//div[contains(@class, 'padding-3')]//a[contains(@href, 'content_')]")
                )
            )
            time.sleep(2)

            links = driver.find_elements(By.XPATH, "//div[contains(@class, 'padding-3')]//a[contains(@href, 'content_')]")
            doc_links = [link.get_attribute("href") for link in links]

            print(f"找到 {len(doc_links)} 个文件链接")

            for doc_url in doc_links:
                try:
                    driver.execute_script("window.open(arguments[0]);", doc_url)
                    driver.switch_to.window(driver.window_handles[-1])
                    print(f"当前正在爬取: {driver.current_url}")
                    time.sleep(3)


                    bulletin_number = None


                    try:
                        title = driver.find_element(By.XPATH, '//td[b[contains(text(),"标　　题：")]]/following-sibling::td').text
                        agency = driver.find_element(By.XPATH, '//td[b[contains(text(),"发文机关：")]]/following-sibling::td').text
                        category = driver.find_element(By.XPATH, '//td[b[contains(text(),"主题分类：")]]/following-sibling::td').text
                        doc_number = driver.find_element(By.XPATH, '//td[b[contains(text(),"发文字号：")]]/following-sibling::td').text
                        pub_date = driver.find_element(By.XPATH, '//td[b[contains(text(),"成文日期：")]]/following-sibling::td').text
                        print("第一种")

                    except:
                        try:
                            title = driver.find_element(By.XPATH, '//td[text()="题："]/following-sibling::td').text
                            agency = driver.find_element(By.XPATH, '//td[text()="发文机关："]/following-sibling::td').text
                            category = driver.find_element(By.XPATH, '//td[text()="主题分类："]/following-sibling::td').text
                            doc_number = driver.find_element(By.XPATH, '//td[text()="发文字号："]/following-sibling::td').text
                            pub_date = driver.find_element(By.XPATH, '//td[text()="成文日期："]/following-sibling::td').text
                            print("第二种")
                        except:
                            try:
                                title = driver.title.replace("_中国政府网", "").strip()


                                bulletin_element = driver.find_element(By.XPATH, '//div[contains(@class, "BreadcrumbNav")]//span[@id="lastNodeID"]')
                                bulletin_number = bulletin_element.text.strip()


                                pub_date_dict = {
                                    "2025年第6号": "2025.2.28", "2025年第5号": "2025.2.20", "2025年第4号": "2025.2.10",
                                    "2025年第3号": "2025.1.30", "2025年第2号": "2025.1.20", "2025年第1号": "2025.1.10",
                                    "2024年第35号": "2024.12.20", "2024年第34号": "2024.12.10", "2024年第33号": "2024.11.30", 
                                    "2024年第32号": "2024.11.20", "2024年第31号": "2024.11.10", "2024年第30号": "2024.10.30", 
                                    "2024年第29号": "2024.10.20", "2024年第28号": "2024.10.10", "2024年第27号": "2024.9.30", 
                                    "2024年第26号": "2024.9.20", "2024年第25号": "2024.9.10", "2024年第24号": "2024.8.30", 
                                    "2024年第23号": "2024.8.20", "2024年第22号": "2024.8.10", "2024年第21号": "2024.7.30", 
                                    "2024年第20号": "2024.7.20", "2024年第19号": "2024.7.10", "2024年第18号": "2024.6.30", 
                                    "2024年第17号": "2024.6.20", "2024年第16号": "2024.6.10", "2024年第15号": "2024.5.30", 
                                    "2024年第14号": "2024.5.20", "2024年第13号": "2024.5.10", "2024年第12号": "2024.4.30", 
                                    "2024年第11号": "2024.4.20", "2024年第10号": "2024.4.10", "2024年第9号": "2024.3.30", 
                                    "2024年第8号": "2024.3.20", "2024年第7号": "2024.3.10", "2024年第6号": "2024.2.29", 
                                    "2024年第5号": "2024.2.20", "2024年第4号": "2024.2.10", "2024年第3号": "2024.1.30", 
                                    "2024年第2号": "2024.1.20", "2024年第1号": "2024.1.10"
                                }
                                pub_date = pub_date_dict.get(bulletin_number, "未知日期")
                                print("第三种")
                            except Exception as e:
                                print(f"无法提取第三种情况的信息，错误: {e}")
                                continue


                    pub_date_formatted = pub_date.replace("年", ".").replace("月", ".").replace("日", "")
                    if bulletin_number:
                        file_name = f"[{pub_date_formatted}]{title}.pdf"
                    else:
                        file_name = f"[{pub_date_formatted}]{title}_{agency}_{category}_{doc_number}.pdf"

                    file_name = sanitize_filename(file_name)
                    file_path = os.path.join(save_dir, file_name)


                    save_pdf(driver, file_path)
                    time.sleep(3)

                    driver.close()
                    driver.switch_to.window(driver.window_handles[0])
                except Exception as e:
                    print(f"无法打开链接: {doc_url}, 错误: {e}")
                    continue
 

            
            try:
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.XPATH, '//button[contains(@class, "btn-next") and not(@disabled)]'))
                )
                driver.execute_script("arguments[0].scrollIntoView();", next_button)
                time.sleep(1)
                next_button.click()
                time.sleep(5)
                print(f"翻页成功，当前页面: {driver.current_url}")
            except Exception as e:
                print("已到最后一页或翻页失败:", e)
                break

        except Exception as e:
            print("页面爬取过程中发生错误:", e)
            break

scrape_page()
driver.quit()
print("爬取完成")

爬取当前页面的所有文件...
找到 15 个文件链接
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202502/content_7004410.htm
第一种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.02.17]国务院办公厅关于转发商务部、国家发展改革委《2025年稳外资行动方案》的通知_国务院办公厅_商贸、海关、旅游&对外经贸合作_国办函〔2025〕16号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202502/content_7003572.htm
第一种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.02.10]国务院关于《武汉市国土空间总体规划（2021—2035年）》的批复_国务院_国土资源、能源&其他_国函〔2025〕23号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202502/content_7003025.htm
第一种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.13]公共安全视频图像信息系统管理条例_国务院_工业、交通&信息产业（含电信）_国令第799号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202502/content_7002337.htm
第一种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.27]国务院办公厅关于推动成品油流通高质量发展的意见_国务院办公厅_国土资源、能源&石油与天然气_国办发〔2025〕5号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202502/content_7002252.htm
第一种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.27]国务院关于同意《海南自由贸易港自驾游进境游艇管理若干规定》的批复_国务院_商贸、海关、旅游&其他_国函〔2

当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202502/content_7005745.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.02.18]国家金融监督管理总局关于港澳银行内地分行开办银行卡业务有关事项的通知_金融监管总局_财政、金融、审计&银行_金规〔2025〕4号.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11886/202502/content_7007130.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.2.28]国家发展改革委 国家数据局关于印发《公共数据资源登记管理暂行办法》的通知　　公共数据资源登记管理暂行办法__2025年第6号国务院公报.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11886/202502/content_7007141.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.2.28]国务院关于《武汉市国土空间总体规划（2021—2035年）》的批复__2025年第6号国务院公报.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11866/202502/content_7004031.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.2.20]中华人民共和国公安部令（第172号）　　公安部关于修改《机动车驾驶证申领和使用规定》的决定　　机动车驾驶证申领和使用规定__2025年第5号国务院公报.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11866/202502/content_7004030.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.2.20]国家金融监督管理总局令（2024年第7号）　　金融机构合规管理办法

当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_6996730.htm
第一种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.02]国务院办公厅关于促进政府投资基金高质量发展的指导意见_国务院办公厅_财政、金融、审计&财政_国办发〔2025〕1号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_6996114.htm
第一种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.12.30]国务院办公厅关于严格规范涉企行政检查的意见_国务院办公厅_综合政务&其他_国办发〔2024〕54号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202502/content_7004206.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.17]国家能源局关于印发《分布式光伏发电开发建设管理办法》的通知_国家能源局_国土资源、能源&电力_国能发新能规〔2025〕7号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202502/content_7004200.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.02.11]国家中医药管理局综合司关于印发《国家中医药管理局主责国家重点研发计划重点专项管理实施细则》的通知_国家中医药管理局综合司_卫生、体育&其他_国中医药综科技函〔2025〕28号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202502/content_7004201.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.12.19]国家文物局办公室 自然资源部办公厅 农业农村部办公厅关于加强大遗址保护规划和用地保障的通知_国家文物局办公室 自然资源部办

PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.02.10]国家移民管理局关于实施东盟国家旅游团入境云南西双版纳免签政策的公告_国家移民局_商贸、海关、旅游&其他_2025年第1号.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11846/202502/content_7002782.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.2.10]人力资源社会保障部 国家发展改革委 工业和信息化部 商务部 全国工商联关于加强人力　资源服务助力制造业高质量发展的意见__2025年第4号国务院公报.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11846/202502/content_7002792.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.2.10]国务院关于《佛山市国土空间总体规划（2021—2035年）》的批复__2025年第4号国务院公报.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11846/202502/content_7002777.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.2.10]中华人民共和国海关总署公告（2024年第168号）　　中华人民共和国海关计核涉嫌走私的货物、物品偷逃税款办法__2025年第4号国务院公报.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11846/202502/content_7002784.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.2.10]中央组织部 人力资源社会保障部 外交部 教育部 科技部 公安部 中国人民银行 海关总署　国家医保局 国家移民局关于进一步做好留学人才回国服务工作的意见__2025年第4号国务院公报.pdf
当前正在爬取: https://w

当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_7001569.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.07]关于有序推进省内异地就医住院费用纳入按病种付费管理的通知_国家医保局办公室 财政部办公厅_卫生、体育&卫生_医保办函〔2025〕3号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_7001563.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.27]国家税务总局关于支持跨境电商出口海外仓发展出口退（免）税有关事项的公告_税务总局_财政、金融、审计&税务_国家税务总局公告2025年第3号.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11826/202501/content_7001293.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.1.30]国务院关于《常州市国土空间总体规划（2021—2035年）》的批复__2025年第3号国务院公报.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11826/202501/content_7001310.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.1.30]中共中央 国务院关于深化养老服务改革发展的意见__2025年第3号国务院公报.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11826/202501/content_7001302.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.1.30]国务院关于《贵阳市国土空间总体规划（2021—2035年）》的批复__2025年第3号国务院公报.pdf
当前正在爬取: https://www.

当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_7000635.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.09]关于开展2024年度政府部门财务报告编报工作的通知_财政部_财政、金融、审计&财政_财库〔2025〕3号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_7000615.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.09]商务部等5部门办公厅（室）关于做好2025年度电动自行车以旧换新工作的通知_商务部办公厅 工业和信息化部办公厅 生态环境部办公厅 市场监管总局办公厅 国家消防救援局办公室_工业、交通&机械制造与重工业_商办流通函〔2025〕10号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_7000414.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.14]教育部办公厅关于印发《中小学科学教育工作指南》的通知_教育部办公厅_科技、教育&教育_教监管厅〔2025〕1号.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11806/202501/content_6999373.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.1.20]中华人民共和国海关总署令（第275号）　　海关总署关于废止《中华人民共和国海关计核涉嫌走私的货物、物品偷逃税款暂行办法》　　　的决定__2025年第2号国务院公报.pdf
当前正在爬取: https://www.gov.cn/gongbao/2025/issue_11806/202501/content_6999381.html
第三种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.1.2

找到 10 个文件链接
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_6999416.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.14]工业和信息化部办公厅关于组织开展2025年未来产业创新任务揭榜挂帅工作的通知_工业和信息化部办公厅_工业、交通&信息产业（含电信）_工信厅高新函〔2025〕21号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_6999289.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.09]自然资源部关于加强地质资料管理的通知_自然资源部_国土资源、能源&其他_自然资规〔2025〕1号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_6999194.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.14]人力资源社会保障部等8部门关于开展2025年春风行动的通知_人力资源社会保障部 交通运输部 农业农村部 全国总工会 共青团中央 全国妇联 中国民航局 国铁集团_劳动、人事、监察&劳动就业_人社部函〔2025〕5号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_6999163.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.09]国家医疗保障局办公室关于推进基本医保基金即时结算改革的通知_国家医保局办公室_卫生、体育&医药管理_医保办发〔2025〕1号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_6999132.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.12.24]关于推进生育

当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_7001718.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.12.31]民政部 中央社会工作部 全国工商联关于加强异地商会登记管理服务工作的通知_民政部 中央社会工作部 全国工商联_市场监管、安全生产监管&工商_民发〔2024〕74号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_6997526.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.12.25]关于开展儿童友好医院建设的意见_国家卫生健康委办公厅 国家发展改革委办公厅 教育部办公厅 财政部办公厅 国家医保局办公室 国家中医药局综合司_卫生、体育&其他_国卫办妇幼发〔2024〕32号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_6997129.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2025.01.05]关于2025年加力扩围实施大规模设备更新和消费品以旧换新政策的通知_国家发展改革委 财政部_财政、金融、审计&其他_发改环资〔2025〕13号.pdf
翻页成功，当前页面: https://sousuo.www.gov.cn/zcwjk/policyRetrieval
爬取当前页面的所有文件...
找到 5 个文件链接
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501/content_6997118.htm
第二种
PDF 已保存: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.12.18]关于依托预算管理一体化系统建立全国行政事业单位国有资产调剂共享平台的通知_财政部_财政、金融、审计&财政_财资〔2024〕175号.pdf
当前正在爬取: https://www.gov.cn/zhengce/zhengceku/202501